In [2]:
import sys # aby importy z dołu działały
sys.path.append(r"C:\Users\Admin\OneDrive\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl")

In [3]:
from scraper_wiki_main import parsuj_misje_z_url, dekompresuj_html, pobierz_soup
from moduly.services_persist_wynik import przefiltruj_dane_misji, normalizuj_nazwe_npc
from moduly.db_core import utworz_engine_do_db
import json
import copy
from bs4 import BeautifulSoup
import zlib
import base64
from sqlalchemy import text
import re
import json
import pandas as pd
from datetime import datetime

In [4]:
silnik = utworz_engine_do_db()

In [5]:
s = "eJztPdtyG7mx7/yKNrcce6vE+0UUJY3Kt911YjuO7Y03SaVY4EyTgxUGmAAYUsxTPiEP5wP8Lf6UfMmpxlxJjmhp195KduUHmUT3AI1Go9E3DM8CvgJfMGPOmyv0rdKtuQo2TeDBeZM+PVHSorRNr3FWQZUq1lzaFM1wi2+TedP7RqsI3jPta7aw8J5f8rNOwFfZo4Tqp7057Oth/X3gj0kUt6xqSbYqgSwnJ1q3HILg8rIJocbFefOraN0KkQVN7/dJFINVINmKL5nlSp512CcfN8i0Hz6XcWLLLtLG9PEKO6K141orm0LKlmidf29ZvLJNb+eJHCishmjdipk2qFsqsTQiBFyfN4XVTRBMLs+bKJvemWVzgeCjECZmPqf2QTPvksuFmqsrCJi+TBH/kaCxc3XVBGM3AmlBdYC6lT08HcRXp2se2HDa72OUfYR+tz3C6PSUxqNZeWdW52OwuVqhYytqgofgK6JFnjf7xSg02xYTfCmnPkqL+nShpG0Z/k+c9vqj++nXNfJlaKdzJYLTmAUBETSKr2hY6jDvLOAmFmwzdTPKSOx1u/dPs7n4SggWG5zmH053Jtk92GNLq/VhBOL26Qq15T4T2bQiHgQCc7Jb2s2EiM/oq0wjY1zTO2OZZHXW/JJ3vmE+SWITLLc06CtMrGai6Z3xaAlM2LIJAmZZa8EFcZ6GOm8O+tVWN+p5szdpQoC+IprOm8xspN+E6hNCsRQm2D83TTDaP292eMSWaDrZYLNBvx3L5QXDMY67TSi77ngk9mcdmlXx32dgW98x6w03XC7hrVU62hkj/8+G9EeTODrJM76K8byp1bpO8gQu7Omc+ZdLrRIZkJwoPbWaSRMzjdKeNr23lmmbdmwDpxCqK/RC6fv9YxtixPTsXYi6slovlH6QQiCDeHtNjl1VGbDWYhQLZpEUDfhK6cA4pQNS4ZWPOrbZsqYwooRtYDhqD+G42x7vCdFbLlaoI6Xk7Am3m4K8sh1cu3dmktj7G3V09PEDdfX3sw41VZe0bi2lkrgryn9VEl+yuGB6rAwnUZ5qFMzyVb5LB91ufHWayt+0777k3c6F8i+rol4oMalszJbohLJO8Psjkso90R9Mht0Dwt/vdg9Lvw2TaN55r7QIXrK4VfLv2R/aP8bLjptL61r4hT8ejU5Oiu0y6HZpv1Q5WnCJzY0SicVTktApLcn9U6viKa3K/VPBZT7Zafc0YnrJ5bQ1ao/iq5/Er16vTlH0DvCKgIdY9Vjw2KkI7OF4zEoV0XMqon7rBp976z6Twd3Gvdu4dxv3f23jvsAVinLrTiatk+6XGOcJs7hUenOtlrjxFtxpSHfeF9BoVzFqjtLHkuRe/2gy+iLseYNrpgNT4U5VU0g118gugf3IrlRiUVvb9PoDh7Oryl4qiZuv3m1iNDO1mPmKS1Mwc1mV/WWtXqjTCq7xWjkf3kQlfKtEQNLe6Q3jq1b+7cLvTwaTcr/3htt2LQy7t5+kqU7S/JKTTEWznGb5/YItjifd4NqJfsEd/lrjiqukkK1cqJw3akKld4MJsWBcCm5ceyK8M8G3Dy73pHPO69qVXLZijQt+BYTDfTrUK15UrXYd1WnX0aEFGd3Mf+qNUvYfnwTzio4dNTvexw9/mwxak8nfP37I2b8tZW8wVtqa2Ru0iZYYFBKWAaAAbPOBtql1DIIKq3YfKpa8EIKjs47gXuM3y+8XSkVcLmdvQxaodbmhs3bI22/C7Z1napgNjGzW3zTD34U4+70iadzMnsmg5Pi7ECEDgAPchOW7D+3z/LTC7yLaJ3Hd3KLqz4oHhgIPs4eu968v0tjMOQbc/o5F8anGgIY87zVBo6DzcaGEIBWZ0V90AVkX8JCsOwgUGpDKAl5xY79uenuIRC08VDENyMTXjuCzTiK8Rhrl/BIa+hVe2cPa+VcqnGRsPr1WOl+yJTcWtbnfP569Q62Zj9PZdypC3+3tYrULxAeQo0EF7Saye7iLfUmuiEEni8p2XKBtK6pslRKWx2Ucus4rhP34bBabzrAXQjGbCtKvPYpJO2wr9DjXndRFmULqmlBDqTsyd22LBVzg9NGKcUHL4ZaYNgGNX/oUFGicVrmStdRtjnEdU8aHNsf4MFPqibs46fnHQYU544w5cBdOuQun3IVTvkA45Ya6xCfhqVckz2SwpUbc919GiexTdTFe9Ca9Ow1yp0HuNMgvqEF+eD2FLDDozJc0kjfN3AX35y6W95uM5aX+YlobE3tn852s+tyDOTkYaIDlJiGwhUUNvopigZaQmRBgQ40IguxgcB49+CyKGV9KMwUb4q6C/2si7vePH0VMzsi53dDe++pJ9kjB278m4gHhQIHT9Iq2fACa11HdGN8xzWTA9KERcpzqCEXb9ghMBrsjPNLMupPK5EGKouMU9MDk8Yamt9tCvbYbZ53YO4u9Z+RVETM/Y84B1iEXCDbkJnUkgVfXcc2FABbHYuOYl8YniIQA58liMXWkNc4C4Z0FQaYhqErp0ZwLbjdTCkzMHmtkfljdAhkYYs1pyJXiAUqrVbxJ7ZL9/TEe1uwP1/hTTY90f2SUzFJKZjuUdHrj+Kp1GOdiEQzGrLdtr3z8UFpkeGVRSyach5xbZqG1sZl2Ouv1ur1Wa6p9avsq6pgYhTjv9UeD45PJblyINh9xFFKO0t6jNfz44T//+r+PHyhw9TaRaxSCFnGN7BKl23tLxqWxbgWLSFG73N8UAggoNBS74Fe6tEB5IxVQnY3YwIILYVwHxteIEtbchsB2JZF634peNT3ilhO0FTcJE4CLBfo2lek0Xs80Zy3B5igEBvONq2Kzynf1YC4+UoQh/LQEzn3QisYoi+9IvKiqroJt1XIp0A/Rv3Tlatmzu81pT/PEWjIgawysJtgNxcaKRzo74RHfzfia6rqwn9ftVSflZRWQ5qwT9nfM0JxEamx6Z441e0DX2oSF0nWT8s46DmFbi7s8CMUtK705ddzqgVW+QRd2afUqduxXf5z/SM0r3I2eWuXLJJpTxV4vPxv35uEqFMsutiJQWQz1MC39Ki1P0fiau6Dm9cT0DxNT6eP21Ayq1GQm0vWUDA5TUiRLb0vFsErFa62WGs0BMoaHycg7uD0doyodT7Kz/tDSjA5TUnZxe1qqvtdXr5Q9JK7jw2S4p29PwXHdqhxkx/HNFuYn8WOyRQ2zfjjzQyaXh/gy+QQ91AtkvdyeopMqRc+yM3FG3vUBkk4Ok5R345IENTRVMh6Ns109m1VyO3vOqeiqqqvRWWHfHZFPsYXGZxQmAJ1awhYlrZIBLj+nbZYfknnyeifrlWgmNpWYR9HgfceXITy7oqJyyFvdGewraRJhKWtbyWBVe/0WJVKo+1HEmZk9RiFYVFKdQcFBIYd6f2ZymTAdgFo4G+EFWWDVwbIFydbjkwuxpefrFHa2FO9CZmHNDDAJqUXGfVizFbbhPcIChbN5IvgxMRbmuFB6z/7/gZGPwSyzYTHLH5h4kLZ4xUfHPsES6YcYQIgamDEsERaUrPMrMkOsXO/su5d92DLsn4QsirmSR7BRCURqhbBglMcBGzIJ0QbmZCZHaAzKJWrThucgEQOHbxW4c5+kz5llNeTkK0SuqQ1x5laotNTq1q9c1kx0oojJADVxOwCDCNxhcw0BLlAaNKm7ECoRVGxOboD2irEFx+YbKPj6wOyS+hRXKiE3Z/adMiWNRTO4Zm/7e3W7fFK8ioN79wTOxOovKklnotFHvsLcy6krIrmLg/xm4iC5xkyjZCS2pYBpDLhGv3DvMuXyutQpr5veD69vrQlL627PTMt1YH72MI0Q8mV4421QtdhqTK+s//fkTgZKYhucui1LDbgBXKlL2oILZNqpBYtaK02aCFeoN0oirEO64IQGuD2CaGNQLIBLXyQBBm3IVeDjxMJz8ENlsmCE4AskfWQsk7S2kLirUrjiIlVAMbcussQlcGtgwXxsw1sWiE1aGpGPH7LAKTWRXCV6k6o4RseBc41pSO7jjZmWmZbbNqJjlYuBWO+PkuIlWjn/PLAEyAMjufVCiXLDKOo/389KfPxg2IbirnPv55xVDwwwa5l/aWCp1RqM1YqOjns1bn+Fct/H2P5sun8n7OkrFuHvlvY0O9PoBF6ipSV0YYj8xBHu0pch4dqKXhirGd+KXlSPjOwkaV8/mXoTKLBlqOMG+5c6maWdzPbMrboRDpheNZHBF8qGml1VKx0qTV75Oe1AIwjlM4sB2RwMKFO1oBDs58hhDSbtCQyP292fm8Oijo4+fqCu7nJY/x05LFqSNIdFq/LrzGGlNr5U2oa5FbtXJchoOpzJ2feSr1CbqjSXUKhAvdrmqrG5FfmuU5R1CqGqK6tqrTxbNbFhK3Cb6cb6sG2pKh4YWCpjeExK1MVrWYTAUtf0t1gc+heVzP4g1Xr2LuRm9mzFxf3BN8Wyk6lPUCAoEPTiRoV2Nc/tV9cV/t3bmFwE55vlIlBkTso4csCZUMsEYatyPY3J5yejzyQZN1SJx2WSLiylbJzDSyfEHJdcunPTKlgwTlk3d5wygRGzWFpcbxU5nGukPgGvYvQrZyw9HZO5SXKzUDq3kOjcLKTgWydov0yhSDpWKtpFpQgbB+Oqi5JlXrbE2W9pprHpPfxTVhjrlubjh4oBo/EfCde030C76vZ8c+dvC/i5e5xW0HCbuEQFbUwyNu7RkqX5NWd1AZMbCFHEbXi2QgkLeu3BNbWks++jueb+XukoZO3eToPTWFXN8UnrLtcn1Wm8UvfgnbPQXRKK7lv7NnU8SChVosEwsXKzTA26XYNnK+qQRRncf87CyTSWkmIDZJPQmxGsirdUJNn/B8y+2kjGnt33swNetc35HHIjjaWKe9ecCp1ZO3tLyZ3ClkobwTV61W+fs0jpGIbj9vAzFCkdk4E3bg/vDLz/DgOPliQz8MZUrvSrNPAYGSWGB3s76kkemZz9QenEhrOXKpGWcbmgIzSX5gILUizYwvIOgtPajmuH/TNKFGlIdS6UKjPgZacOBSoo3vWw2tHeMz17EjJnGszeoix1x3umIYeAg3h7TWVxCkWujaXQ8YGrK4/dOzSM1TxAfftrK1uPX3dlZQvpVhZ1rfKtnlOPhCgDDM54MXncIbOj7sErta4LhcTet1olsaHz4wCDvqWuyZOohtNnD3c06U+48ZN3vJ1Jebijiq+9B5Q/bnbOvxqB+o7RwAsmxOwRanUpt87mEgoVqFfbbK4tfUrJnn3HZDB7SjkCer/P9vECBIQC6NW1ugGcmTpHCiVK0Elq48bM2COwisL4dYftcyOQ1uhPCVLo7CmTrAxGE5D4RMAHKcjbb9tK1DwKAqd/qSbmCCQyPd8ckpNtgeD/RHl7kdhee+rjutXfx0w5p3yfUYzYVfKYUCV2+qm9tnG5qrN5TZ8fPxCwYtnmvolhKwwgMfeAgrkLpSFUayCtfXED0/M2IzrzkxvKSXG5vAdPWJqDi5mgI8ekKSi6rpbGep1P/TmJeI+OjyIAXDE/cX4VTXodokaw6qJWtbTbbRLrXJBjdYkSrLqhK1gHb6WFR673fZK3vZjYew4hXrNRqFiSz6zmcyyrE6kN0jbPfbkukPluzQVpAlfk+FiwoOwkBz0wkAI8PxGWm1RrON8hXcXU33ERC5XYsA3PnWtrlXLpQydPidPjOcsv8l353nF97dZjjaQcaAl2fLgvzdz32G63afhMMJil2HYe9Q4R/JCLQKMEldjSC3/JNnOExCCQ48mEuShAexllzSJW0Z/Zdy/7kG71NRrniMWoYkG8jNCGXC53XNqtmETtgXqLAEWevi7U5MsNWB6hi+PT3c82vKdtGCiXC6DEdbF2d9GEPfvmoClatXOeL1z4xombC39adknZM4SFMxusKl1nivQ8CGDJV5jagIlP9ZuBTiK3+0xI4FDFqdrkss7SriSgjtJ9nhtAjwXzL2fPopjrcu9XBnfIuVHikCFD9pyGvpR8gbeNUnzSEKysSqxVxA2VbvC9ioPi1uz+VdzKldq0WtoxO9ZqRexh4LwwF+54TiqOIoJKg6TkY515eeN8r0sA1RWAuVSj+nW/zYJuyG7Fc0tuUfbYxTryDzd7Z0XdawArCf5f+9sT/pcvqP8W1ubapXmlZt8g07mWfcr0ZXkpXgHBcqXqYDdZgprn7vi+a2uxFc7Y7DVH3x1xL7ksozoEBAYOSFx0wJtwvu7BfdaDcwvhN8n3R7OXzFJ+Qy1mb63mC5ylsYNqycsjSJGIhSmS80lSpJusw6EO7rZCzct1vmH6iP7MvtGUeKwEccgbJyj9gQJ609fs7D15x/3dYlgm6XXTM/LTdrZBBkpduJsL//5jdypoWwW9pdcaLfFo9pQvFkgvN5q5nE1FAeUoUKBAinIz9XPd43fiv2//vNZsg3q2UNqZQO+5f1l5hd0rBSmCiw+RPZMh3NASqn34mmU466ib+XDbt23qr804P26roHwnaUoBFoktl+Q8hYjLdFGmMKIc6P7blnZmTIsOkWyl3ezlfJ240Smb/67CdPaSB9s5gKKhIjsl0r4Ijequ6Pa6B1Oq/ZtkVPNBs8m4Iu4hpVJrABeTbm84nhR55Z13xUP6svhCDIfd2mJvOJvvcixd1V6/3W33Chala5s1etVv6QvY+t3+uNXtt3rdr7PUVOCqngHOSJ549vKW21SE796cuuYKVCZfxR3kDLt6TSu7duxk46dcVXY76fyk3x3v5S689yle7fUww5ea422HDebloKZz3ajvnz7eu/51r9WCxitcv34NgkfcZmG5xhPm7vJQpJJ+nqI/7g56o+64P+wPUhhVRXG9mUJ/dNLvdruNNxgkPgZF+4IJgw1XO899V/ZipvA3E6r1f/71b6v8vzeevP4+DYUmhi1xCt12b3AMBn0lA9N4gzTnLfBg0i/ArzXGWvlojNJ0fZpTVYlUgYv5STuF/vjkuNPrun+NN0goSoL7RQwYjk76nWHvZDjoDmG+sWgar5Wx//nXv/EqJnM3q8HP0Ae948FkB/9dVmgCTC+TiE6pFLfXG4+HO7hUAkV3fFznjowAY1JXvRFR2KD3EEtD0cf0F0lgkUiXCcvn0uuMut3G95IywjHdvEl0tZtup19C462JpER1c0Z0M4qeXdkXSsUmH6BLA3QbLxK2w/Jud9zpjdr0ZM55QoowUnqTo41Ho35v1Bn1h/3JpBjjlbumSH5EtlAYwBOml4qOGXpbwBS6jXc0momJfxkW2V+7aOn48JCtlu3829eVAVJ8KuXn0pBSoxpwVwtkzRR6jVYrFfXGO3opoy8Sx71yOdycU8mHh/ePInNEN/vNUV5N9HWj16Vh7wP0+yft7skxuH89aFllmWjQG466g/sAvXG3PRwMc3AuJVMXEp+rqwaMe+2ewxx2293hPubz9DdlGtA7bg9oRBictMf9SYrYLxGfKGMb0Ou1B9Qd9Eftk9HxHpZ7s14DYNSejAmtN2gPe+MUbViiPc3SFw2AYfvkxGH2yplUMN9pHjmsQY+w4KQ9mXRTrEGVuAbAoH3iSINJ++RkfwLZmwyf+0o65KEjEI7bJ4PJHlte52mVYjGBQga0VfNd4zu95Gz3S6QqufT0nqGcxg425UHITDgdj4bDSb/VveczqSSdgOn9HB6hsSyKtxVe3wHpDctOVngA4/Fxv9vrteGNS8E7j4EZygYul6gxoHe+sMTgFCiT3aKEdAMc2fsJN/ejTgulyEpJLQSpjORxjFSE5L1BqznSPF0KsHnGKi9S2DkLsvm26UhuL5fZC45deHfmwrsXSgQ8OM/Ib3o/9Uk6RZrb776pzMhnzoTMb7BFa/qdJIuaLiGlx3SBsfv7TFLpiIlW2UP2cojd9r2QVIw+Z2KavYGeY5mtqAHlL6rnZHIybwrXXCLO32c/3SldmaXnbFnJtI/nKlFSc9rsXV/O2sv7bgeGzh2F64bM4MVQ29/PbjKEU01mxuzM+Rv7Y6QIlLRNEbydhhuN4oy7GZezuM5W3EZL93TVeqxt3r4wuC2P7r//B6+yS6k="

In [6]:
html = dekompresuj_html(s)
wynik = parsuj_misje_z_url(None, html_content=html)

In [7]:
wynik = json.dumps(wynik, indent=2, ensure_ascii=False)

In [8]:
print(wynik)

{
  "Źródło": {
    "url": null,
    "html_skompresowany": "eJztPdtyG7mx7/yKNrcce6vE+0UUJY3Kt911YjuO7Y03SaVY4EyTgxUGmAAYUsxTPiEP5wP8Lf6UfMmpxlxJjmhp195KduUHmUT3AI1Go9E3DM8CvgJfMGPOmyv0rdKtuQo2TeDBeZM+PVHSorRNr3FWQZUq1lzaFM1wi2+TedP7RqsI3jPta7aw8J5f8rNOwFfZo4Tqp7057Oth/X3gj0kUt6xqSbYqgSwnJ1q3HILg8rIJocbFefOraN0KkQVN7/dJFINVINmKL5nlSp512CcfN8i0Hz6XcWLLLtLG9PEKO6K141orm0LKlmidf29ZvLJNb+eJHCishmjdipk2qFsqsTQiBFyfN4XVTRBMLs+bKJvemWVzgeCjECZmPqf2QTPvksuFmqsrCJi+TBH/kaCxc3XVBGM3AmlBdYC6lT08HcRXp2se2HDa72OUfYR+tz3C6PSUxqNZeWdW52OwuVqhYytqgofgK6JFnjf7xSg02xYTfCmnPkqL+nShpG0Z/k+c9vqj++nXNfJlaKdzJYLTmAUBETSKr2hY6jDvLOAmFmwzdTPKSOx1u/dPs7n4SggWG5zmH053Jtk92GNLq/VhBOL26Qq15T4T2bQiHgQCc7Jb2s2EiM/oq0wjY1zTO2OZZHXW/JJ3vmE+SWITLLc06CtMrGai6Z3xaAlM2LIJAmZZa8EFcZ6GOm8O+tVWN+p5szdpQoC+IprOm8xspN+E6hNCsRQm2D83TTDaP292eMSWaDrZYLNBvx3L5QXDMY67TSi77ngk9mcdmlXx32dgW98x6w03XC7hrVU62hkj/8+G9EeTODrJM76K8byp1bpO8gQu7Omc+ZdLrRIZkJwoPbWaSRMzjdKeNr23lmmbdmwDpxCqK/RC6fv9YxtixPTsXYi6slovlH6QQiCDeHtNjl1VGbDWYhQLZpEUDfhK

In [9]:
wynik = json.loads(wynik)

In [10]:
start_NPC = normalizuj_nazwe_npc(wynik["Misje_EN"]["Podsumowanie_EN"]["Start_NPC"])
koniec_NPC = normalizuj_nazwe_npc(wynik["Misje_EN"]["Podsumowanie_EN"]["Koniec_NPC"])

In [20]:
q_select_id_puste_npc = text("""
    SELECT MISJA_ID_MOJE_PK
    FROM dbo.MISJE
    WHERE 1=1
      AND (NPC_START_ID IS NULL OR NPC_KONIEC_ID IS NULL)
      AND MISJA_ID_Z_GRY <> 123456789
""")

In [21]:
q_select_npc = text("""
    SELECT NPC_ID_MOJE_PK
    FROM dbo.NPC
    WHERE NAZWA = :npc
""")

In [ ]:
q_select_html = text("""
SELECT 
	MISJA_ID_MOJE_FK, 
    HTML_SKOMPRESOWANY,
	MAX(DATA_WYSCRAPOWANIA) AS M
FROM dbo.ZRODLO
WHERE MISJA_ID_MOJE_FK = :m
GROUP BY MISJA_ID_MOJE_FK, HTML_SKOMPRESOWANY
;
""")

In [30]:
with silnik.connect() as conn:
    #w = conn.execute(q_select_npc, {"npc": start_NPC}).scalar_one()
    misje_puste_npc = conn.execute(q_select_id_puste_npc).scalars().all()
    for id_misji in misje_puste_npc:
        html = conn.execute(q_select_html, {"m": id_misji}).scalar_one()
        wynik = parsuj_misje_z_url(None, html_content=html)


TypeError: Incoming markup is of an invalid type: 3. Markup must be a string, a bytestring, or an open filehandle.